In [33]:
!pip install kafka-python pandas

In [34]:
%pip install apache-flink==1.18.1

Note: you may need to restart the kernel to use updated packages.


In [42]:
from kafka.admin import KafkaAdminClient, NewTopic

admin = KafkaAdminClient(bootstrap_servers='kafka:29092')
try:
    admin.create_topics([NewTopic(
        name='sales_topic',
        num_partitions=1,
        replication_factor=1
    )])
except Exception as e:
    print(f"Err: {e}")

print(admin.list_topics())

Err: [Error 36] TopicAlreadyExistsError: Request 'CreateTopicsRequest_v3(create_topic_requests=[(topic='sales_topic', num_partitions=1, replication_factor=1, replica_assignment=[], configs=[])], timeout=30000, validate_only=False)' failed with response 'CreateTopicsResponse_v3(throttle_time_ms=0, topic_errors=[(topic='sales_topic', error_code=36, error_message="Topic 'sales_topic' already exists.")])'.
['sales_topic']


In [43]:
from kafka.admin import KafkaAdminClient, NewTopic

admin = KafkaAdminClient(bootstrap_servers='kafka:29092')
admin.delete_topics(['sales_topic'])

import time
time.sleep(3)

admin.create_topics([NewTopic(
    name='sales_topic',
    num_partitions=1,
    replication_factor=1
)])
print("Topic recreated", admin.list_topics())

Topic recreated ['sales_topic']


In [44]:
import pandas as pd
import json
import glob
from kafka import KafkaProducer

producer = KafkaProducer(
    bootstrap_servers=['kafka:29092'],
    value_serializer=lambda v: json.dumps(v).encode('utf-8'),
    acks='all',
    retries=3
)

topic_name = 'sales_topic'
csv_files = glob.glob("/home/jovyan/work/data/*.csv")

for file in csv_files:
    print(f"Reading file: {file}")
    df = pd.read_csv(file)
    
    # NaN to Null
    df = df.where(pd.notna(df), other=None)
    
    records = df.to_dict(orient='records')
    for record in records:
        producer.send(topic_name, value=record)
    producer.flush()
    print(f"File {file} sent: {len(records)} records")

producer.close()
print("Done")

Reading file: /home/jovyan/work/data/MOCK_DATA (1).csv
File /home/jovyan/work/data/MOCK_DATA (1).csv sent: 1000 records
Reading file: /home/jovyan/work/data/MOCK_DATA (2).csv
File /home/jovyan/work/data/MOCK_DATA (2).csv sent: 1000 records
Reading file: /home/jovyan/work/data/MOCK_DATA (3).csv
File /home/jovyan/work/data/MOCK_DATA (3).csv sent: 1000 records
Reading file: /home/jovyan/work/data/MOCK_DATA (4).csv
File /home/jovyan/work/data/MOCK_DATA (4).csv sent: 1000 records
Reading file: /home/jovyan/work/data/MOCK_DATA (5).csv
File /home/jovyan/work/data/MOCK_DATA (5).csv sent: 1000 records
Reading file: /home/jovyan/work/data/MOCK_DATA (6).csv
File /home/jovyan/work/data/MOCK_DATA (6).csv sent: 1000 records
Reading file: /home/jovyan/work/data/MOCK_DATA (7).csv
File /home/jovyan/work/data/MOCK_DATA (7).csv sent: 1000 records
Reading file: /home/jovyan/work/data/MOCK_DATA (8).csv
File /home/jovyan/work/data/MOCK_DATA (8).csv sent: 1000 records
Reading file: /home/jovyan/work/data/MOC

In [35]:
import os
!mkdir -p /home/jovyan/work/jars

!wget -nc -P /home/jovyan/work/jars \
  https://repo.maven.apache.org/maven2/org/apache/flink/flink-sql-connector-kafka/1.17.0/flink-sql-connector-kafka-1.17.0.jar

!wget -nc -P /home/jovyan/work/jars \
  https://repo1.maven.org/maven2/org/apache/flink/flink-connector-jdbc/3.1.2-1.18/flink-connector-jdbc-3.1.2-1.18.jar

!wget -nc -P /home/jovyan/work/jars \
  https://repo1.maven.org/maven2/org/postgresql/postgresql/42.6.0/postgresql-42.6.0.jar

!ls -la /home/jovyan/work/jars/

File ‘/home/jovyan/work/jars/flink-sql-connector-kafka-1.17.0.jar’ already there; not retrieving.

File ‘/home/jovyan/work/jars/flink-connector-jdbc-3.1.2-1.18.jar’ already there; not retrieving.

File ‘/home/jovyan/work/jars/postgresql-42.6.0.jar’ already there; not retrieving.

total 12460
drwxrwxrwx 1 jovyan 1000    4096 May 11 11:14 .
drwxrwxrwx 1 jovyan 1000    4096 May 11 11:53 ..
-rwxrwxrwx 1 jovyan 1000  266420 Jun 15  2023 flink-connector-jdbc-3.1.1-1.17.jar
-rwxrwxrwx 1 jovyan 1000  268555 Feb  1  2024 flink-connector-jdbc-3.1.2-1.18.jar
-rwxrwxrwx 1 jovyan 1000 5563429 Mar 17  2023 flink-sql-connector-kafka-1.17.0.jar
-rwxrwxrwx 1 jovyan 1000 5563764 Nov 10  2023 flink-sql-connector-kafka-1.17.2.jar
-rwxrwxrwx 1 jovyan 1000 1081604 Mar 17  2023 postgresql-42.6.0.jar


In [18]:
!cp /home/jovyan/work/jars/*.jar /opt/conda/lib/python3.10/site-packages/pyflink/lib/

In [36]:
import os
from pyflink.table import TableEnvironment, EnvironmentSettings

env_settings = EnvironmentSettings.in_streaming_mode()
t_env = TableEnvironment.create(env_settings)

In [38]:
t_env.execute_sql("""
   CREATE TABLE IF NOT EXISTS kafka_source (
        id INT,
        customer_first_name STRING,
        customer_last_name STRING,
        customer_age INT,
        customer_email STRING,
        customer_country STRING,
        customer_postal_code STRING,
        customer_pet_type STRING,
        customer_pet_name STRING,
        customer_pet_breed STRING,
        seller_first_name STRING,
        seller_last_name STRING,
        seller_email STRING,
        seller_country STRING,
        seller_postal_code STRING,
        product_name STRING,
        product_category STRING,
        product_price DECIMAL(10,2),
        product_quantity INT,
        sale_date STRING,
        sale_customer_id INT,
        sale_seller_id INT,
        sale_product_id INT,
        sale_quantity INT,
        sale_total_price DECIMAL(10,2),
        store_name STRING,
        store_location STRING,
        store_city STRING,
        store_state STRING,
        store_country STRING,
        store_phone STRING,
        store_email STRING,
        pet_category STRING,
        product_weight STRING,
        product_color STRING,
        product_size STRING,
        product_brand STRING,
        product_material STRING,
        product_description STRING,
        product_rating DECIMAL(3,2),
        product_reviews INT,
        product_release_date STRING,
        product_expiry_date STRING,
        supplier_name STRING,
        supplier_contact STRING,
        supplier_email STRING,
        supplier_phone STRING,
        supplier_address STRING,
        supplier_city STRING,
        supplier_country STRING
    ) WITH (
    'connector' = 'kafka',
    'topic' = 'sales_topic',
    'properties.bootstrap.servers' = 'kafka:29092',
    'properties.group.id' = 'student_group',
    'format' = 'json',
    'scan.startup.mode' = 'earliest-offset'
    )
""")

In [ ]:
t_env.execute_sql("""
    CREATE TABLE pg_dim_suppliers (
        supplier_id INT,
        name STRING,
        contact STRING,
        email STRING,
        phone STRING,
        address STRING,
        city STRING,
        country STRING,
        PRIMARY KEY (supplier_id) NOT ENFORCED
    ) WITH (
        'connector' = 'jdbc',
        'url' = 'jdbc:postgresql://postgres:5432/lab3_db',
        'table-name' = 'dim_suppliers',
        'username' = 'postgres',
        'password' = 'password',
        'sink.parallelism' = '1' 
    )
""")

t_env.execute_sql("""
    CREATE TABLE pg_dim_brands (
        brand_id INT,
        name STRING,
        PRIMARY KEY (brand_id) NOT ENFORCED
    ) WITH (
        'connector' = 'jdbc',
        'url' = 'jdbc:postgresql://postgres:5432/lab3_db',
        'table-name' = 'dim_brands',
        'username' = 'postgres',
        'password' = 'password',
        'sink.parallelism' = '1' 
    )
""")

t_env.execute_sql("""
    CREATE TABLE pg_dim_stores (
        store_id INT,
        name STRING,
        location STRING,
        city STRING,
        state STRING,
        country STRING,
        phone STRING,
        email STRING,
        PRIMARY KEY (store_id) NOT ENFORCED
    ) WITH (
        'connector' = 'jdbc',
        'url' = 'jdbc:postgresql://postgres:5432/lab3_db',
        'table-name' = 'dim_stores',
        'username' = 'postgres',
        'password' = 'password',
        'sink.parallelism' = '1' 
    )
""")

t_env.execute_sql("""
    CREATE TABLE pg_dim_customers (
        customer_id INT,
        first_name STRING,
        last_name STRING,
        age INT,
        email STRING,
        country STRING,
        postal_code STRING,
        pet_type STRING,
        pet_name STRING,
        pet_breed STRING,
        PRIMARY KEY (customer_id) NOT ENFORCED
    ) WITH (
        'connector' = 'jdbc',
        'url' = 'jdbc:postgresql://postgres:5432/lab3_db',
        'table-name' = 'dim_customers',
        'username' = 'postgres',
        'password' = 'password',
        'sink.parallelism' = '1' 
    )
""")

t_env.execute_sql("""
    CREATE TABLE pg_dim_sellers (
        seller_id INT,
        first_name STRING,
        last_name STRING,
        email STRING,
        country STRING,
        postal_code STRING,
        PRIMARY KEY (seller_id) NOT ENFORCED
    ) WITH (
        'connector' = 'jdbc',
        'url' = 'jdbc:postgresql://postgres:5432/lab3_db',
        'table-name' = 'dim_sellers',
        'username' = 'postgres',
        'password' = 'password',
        'sink.parallelism' = '1' 
    )
""")

t_env.execute_sql("""
    CREATE TABLE pg_dim_products (
        product_id INT,
        name STRING,
        category STRING,
        price DECIMAL(10,2),
        weight STRING,
        color STRING,
        size STRING,
        material STRING,
        description STRING,
        rating DECIMAL(3,2),
        supplier_id INT,
        brand_id INT,
        PRIMARY KEY (product_id) NOT ENFORCED
    ) WITH (
        'connector' = 'jdbc',
        'url' = 'jdbc:postgresql://postgres:5432/lab3_db',
        'table-name' = 'dim_products',
        'username' = 'postgres',
        'password' = 'password',
        'sink.parallelism' = '1' 
    )
""")

t_env.execute_sql("""
    CREATE TABLE pg_fact_sales (
        sale_id INT,
        sale_date DATE,
        customer_id INT,
        seller_id INT,
        product_id INT,
        store_id INT,
        quantity INT,
        total_price DECIMAL(10,2),
        PRIMARY KEY (sale_id) NOT ENFORCED
    ) WITH (
        'connector' = 'jdbc',
        'url' = 'jdbc:postgresql://postgres:5432/lab3_db',
        'table-name' = 'fact_sales',
        'username' = 'postgres',
        'password' = 'password',
        'sink.parallelism' = '1' 
    )
""")

print("Done!")

In [ ]:
statement_set = t_env.create_statement_set()

statement_set.add_insert_sql("""
    INSERT INTO pg_dim_brands (brand_id, name)
    SELECT id, product_brand
    FROM kafka_source
""")

statement_set.add_insert_sql("""
    INSERT INTO pg_dim_suppliers (supplier_id, name, contact, email, phone, address, city, country)
    SELECT id, supplier_name, supplier_contact, supplier_email,
           supplier_phone, supplier_address, supplier_city, supplier_country
    FROM kafka_source
""")

statement_set.add_insert_sql("""
    INSERT INTO pg_dim_stores (store_id, name, location, city, state, country, phone, email)
    SELECT id, store_name, store_location, store_city,
           store_state, store_country, store_phone, store_email
    FROM kafka_source
""")

statement_set.add_insert_sql("""
    INSERT INTO pg_dim_customers (customer_id, first_name, last_name, age, email, country,
                                   postal_code, pet_type, pet_name, pet_breed)
    SELECT sale_customer_id, customer_first_name, customer_last_name, customer_age,
           customer_email, customer_country, customer_postal_code,
           customer_pet_type, customer_pet_name, customer_pet_breed
    FROM kafka_source
""")

statement_set.add_insert_sql("""
    INSERT INTO pg_dim_sellers (seller_id, first_name, last_name, email, country, postal_code)
    SELECT sale_seller_id, seller_first_name, seller_last_name,
           seller_email, seller_country, seller_postal_code
    FROM kafka_source
""")

statement_set.add_insert_sql("""
    INSERT INTO pg_dim_products (product_id, name, category, price, weight, color,
                                  size, material, description, rating, supplier_id, brand_id)
    SELECT sale_product_id, product_name, product_category, product_price,
           product_weight, product_color, product_size, product_material,
           product_description, product_rating, 
           id, -- supplier_id совпадает с ключом поставщика
           id  -- brand_id совпадает с ключом бренда
    FROM kafka_source
""")

statement_set.add_insert_sql("""
    INSERT INTO pg_fact_sales (sale_id, sale_date, customer_id, seller_id, product_id, store_id, quantity, total_price)
    SELECT id, 
           TO_DATE(sale_date, 'M/d/yyyy'), 
           sale_customer_id, 
           sale_seller_id, 
           sale_product_id, 
           id, -- store_id совпадает с ключом магазина
           sale_quantity, 
           sale_total_price
    FROM kafka_source
""")

print("Flink Streaming Job")
statement_set.execute().wait()

In [48]:
!pip install psycopg2-binary

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 3.5 MB/s eta 0:00:0000:0100:01


In [49]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host="postgres",
    port=5432,
    dbname="lab3_db",
    user="postgres",
    password="password"
)

tables = [
    "dim_brands",
    "dim_suppliers",
    "dim_stores",
    "dim_customers",
    "dim_sellers",
    "dim_products",
    "fact_sales"
]

for table in tables:
    df = pd.read_sql(f"SELECT COUNT(*) AS cnt FROM {table}", conn)
    print(f"{table}: {df['cnt'][0]} строк")

conn.close()

dim_brands: 1000 строк
dim_suppliers: 1000 строк
dim_stores: 1000 строк
dim_customers: 1000 строк
dim_sellers: 1000 строк
dim_products: 1000 строк
fact_sales: 1000 строк


/tmp/ipykernel_1015/3405559775.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT COUNT(*) AS cnt FROM {table}", conn)
